# DataFrame Handling Toolkit

A reference notebook of common pandas patterns for exploring, counting, transforming,
filtering, and cleaning DataFrames. Examples use `df`; swap in your own frame.

Sections:
1. [Load & view data](#sec1)
2. [Counting & summarizing](#sec2)
3. [Selecting columns & rows](#sec3)
4. [Filtering rows](#sec4)
5. [map / apply / transform](#sec5)
5b. [Replace & substitute values](#sec5b)
6. [groupby & aggregation](#sec6)
7. [Sorting & ranking](#sec7)
8. [Reusable helper functions](#sec8)
9. [Plot a group-level metric vs a feature](#sec9)
10. [Parsing dates](#sec10)
11. [Scaling & normalization](#sec11)
12. [Character encodings](#sec12)
13. [Inconsistent text entry & fuzzy matching](#sec13)


In [ ]:
# --- Setup: install optional packages used in sections 11-13 (run once) ---
# Uncomment if not already installed. scipy/scikit-learn ship with most envs.
# %pip install chardet rapidfuzz scikit-learn scipy

# Core imports (pandas/numpy are also re-imported in section 1 so it runs standalone)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Optional — only needed for specific sections; safe to skip if unused:
#   section 11 (scaling):   from sklearn.preprocessing import MinMaxScaler, StandardScaler ; from scipy import stats
#   section 12 (encodings): import chardet
#   section 13 (fuzzy):     from rapidfuzz import process, fuzz


<a id="sec1"></a>
## 1. Load & view data

First look at any new dataset. Read string-heavy ID columns (like `zip`) as `str`
to preserve leading zeros.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('Boats_Cleaned_dataset.csv', index_col=0, dtype={'zip': str})

df.shape            # (rows, columns)
df.head(5)          # first 5 rows   (df.tail() for last, df.sample(5) for random)


In [ ]:
df.info()           # dtypes + non-null counts per column (great first look)
df.describe()       # summary stats for NUMERIC columns
df.describe(include='object')   # summary for CATEGORICAL/string columns
df.dtypes           # dtype of each column
df.columns.tolist() # list of column names
def missing_data_summary(df):
    missing_data = pd.DataFrame({
        'Missing_Count': df.isnull().sum(),
        'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2),
        'Data_Type': df.dtypes
    })
    missing_data = missing_data[missing_data['Missing_Count'] > 0].sort_values('Missing_Percentage', ascending=False)
    print(missing_data)


In [ ]:
# Missing values per column (only those with any), as count and percent
miss = df.isna().sum()
miss = miss[miss > 0].sort_values(ascending=False)
pd.DataFrame({'missing': miss, 'pct': (miss / len(df) * 100).round(2)})


In [ ]:
# --- Drop missing values (the counterpart to fillna in section 5b) ---
df.dropna()                      # drop any ROW containing a NaN (often too aggressive)
df.dropna(axis=1)                # drop any COLUMN containing a NaN
df.dropna(subset=['price'])      # drop rows missing a SPECIFIC column
df.dropna(thresh=5)              # keep rows with at least 5 non-NaN values
df.dropna(how='all')            # drop only fully-empty rows

# Always check how much data a drop would cost before committing:
print(df.shape, '->', df.dropna().shape)
print('cols lost if axis=1:', df.shape[1] - df.dropna(axis=1).shape[1])


<a id="sec2"></a>
## 2. Counting & summarizing

`value_counts` is the workhorse for understanding a column's distribution.

In [ ]:
df['boatClass'].value_counts()                 # counts per category (desc)
df['boatClass'].value_counts(normalize=True)   # as proportions (sum to 1)
df['boatClass'].value_counts(dropna=False)     # include NaN as its own row
df['boatClass'].nunique()                       # number of distinct values
df['boatClass'].unique()                        # array of distinct values


In [ ]:
# Binned counts for a NUMERIC column (distribution by range)
df['price'].value_counts(bins=6, sort=False)    # 6 equal-width bins

# Custom ranges with pd.cut
ranges = [0, 10_000, 25_000, 50_000, 100_000, 250_000, df['price'].max()]
pd.cut(df['price'], bins=ranges).value_counts(sort=False)

# Cross-tab: counts of one category vs another
pd.crosstab(df['boatClass'], df['fuelType'])


In [5]:
# Cardinality report — how many distinct values per column (spot 'coarse' cols)
pd.DataFrame({
    'distinct': df.nunique(),
    'pct_unique': (df.nunique() / len(df) * 100).round(2),
    'dtype': df.dtypes.astype(str),
}).sort_values('distinct')


,distinct,pct_unique,dtype
condition,2,0.01,str
type,3,0.02,str
fuelType,4,0.02,str
numEngines,5,0.03,int64
engineCategory,9,0.05,str
hullMaterial,9,0.05,str
created_month,12,0.06,int64
created_year,16,0.08,int64
state,49,0.26,str
minEngineYear,66,0.35,float64


<a id="sec3"></a>
## 3. Selecting columns & rows

`.loc` selects by label, `.iloc` by integer position.

In [ ]:
df['price']                       # one column (Series)
df[['make', 'model', 'price']]    # several columns (DataFrame)

df.loc[10]                        # row by index label
df.loc[10, 'price']               # single cell
df.loc[df['price'] > 50000, ['make', 'price']]   # rows by condition, chosen cols
df.iloc[0:5]                      # first 5 rows by position
df.iloc[0:5, 0:3]                 # first 5 rows, first 3 cols

df.select_dtypes(include='number').columns      # all numeric columns
df.select_dtypes(include=['object', 'category']).columns   # all categorical


<a id="sec4"></a>
## 4. Filtering rows

Build boolean masks and combine with `&` (and), `|` (or), `~` (not).
**Wrap each condition in parentheses.**

In [ ]:
# Single + combined conditions
df[df['price'] > 50000]
df[(df['price'] > 50000) & (df['year'] >= 2015)]
df[(df['state'] == 'FL') | (df['state'] == 'CA')]
df[~(df['fuelType'] == 'other')]                 # NOT

# isin — membership in a list (cleaner than chained OR)
df[df['state'].isin(['FL', 'CA', 'TX'])]

# between — inclusive range
df[df['price'].between(10000, 50000)]

# query — string syntax (often more readable)
df.query('price > 50000 and year >= 2015')
flo = 'FL'
df.query('state == @flo')                        # @ references a Python variable

# string filters (.str accessor)
df[df['make'].str.contains('Sea', na=False)]
df[df['make'].str.startswith('Yam', na=False)]

# null filters
df[df['totalHP'].isna()]
df[df['totalHP'].notna()]


<a id="sec5"></a>
## 5. map / apply / transform

- **`map`** — element-wise on a **Series** (dict lookup or function).
- **`apply`** — a function along a Series, or across DataFrame rows/columns.
- **`transform` (with groupby)** — returns a result aligned to the original index.

> Note: `groupby(KEY).transform(...)` returns NaN for rows where KEY is NaN —
> assigning it back will WIPE those rows. Fill/guard the key first, or use a masked write.

In [ ]:
# map: recode categories with a dict
fuel_simple = {'gasoline': 'gas', 'diesel': 'diesel', 'electric': 'electric', 'other': 'other'}
df['fuel_simple'] = df['fuelType'].map(fuel_simple)

# map: apply a function element-wise
df['price_k'] = df['price'].map(lambda p: p / 1000)

# apply on a Series
df['name_len'] = df['make'].apply(lambda s: len(str(s)))

# apply across ROWS (axis=1) — combine multiple columns
df['hp_per_ft'] = df.apply(
    lambda r: r['totalHP'] / r['length_ft'] if r['length_ft'] else np.nan, axis=1)

# vectorized is FASTER than apply when possible — prefer this:
df['hp_per_ft'] = df['totalHP'] / df['length_ft']

# np.where for conditional columns
df['is_powerboat'] = np.where(df['type'] == 'power', 1, 0)

# np.select for multiple conditions
conds = [df['price'] < 20000, df['price'] < 60000]
labels = ['budget', 'mid']
df['tier'] = np.select(conds, labels, default='premium')


<a id="sec5b"></a>
## 5b. Replace & substitute values

`replace` swaps specific values; `fillna` fills missing; `mask`/`where` replace by
condition; `clip` caps outliers; `.str.replace` edits text.

In [ ]:
# replace exact values
df['fuelType'].replace('other', 'unknown')                 # one value
df['fuelType'].replace(['other', 'na'], 'unknown')         # several -> one
df['fuelType'].replace({'gasoline': 'gas', 'diesel': 'dsl'})  # dict mapping

# replace per-column (outer key = column, inner = value mapping)
df.replace({'fuelType': {'other': 'unknown'},
            'condition': {'used': 0, 'new': 1}})

# IMPORTANT: zeros-as-missing — turn impossible 0s into NaN before imputing
df['totalHP'] = df['totalHP'].replace(0, np.nan)

# regex replace on strings (clean text)
df['make'].str.replace(r'\s+', ' ', regex=True)            # collapse whitespace
df['make'].str.replace('Boats', '', regex=False).str.strip()


In [ ]:
# fillna — replace MISSING values
df['fuelType'].fillna('unknown')                # constant
df['totalHP'].fillna(df['totalHP'].median())    # statistic
df['totalHP'].fillna(df.groupby('boatClass')['totalHP'].transform('median'))  # group stat

# mask / where — replace by CONDITION
df['price'].mask(df['price'] > 1_000_000, np.nan)    # set matches -> value (replace where TRUE)
df['price'].where(df['price'] <= 1_000_000, np.nan)  # keep matches, replace REST (where FALSE)

# clip — cap outliers into a range (a kind of replace)
df['price'].clip(lower=500, upper=500_000)           # values outside are pulled to the bounds

# replace impossible values with NaN via condition (e.g. beam wider than length)
bad = df['beam_ft'] >= df['length_ft']
df['beam_ft'] = df['beam_ft'].mask(bad, np.nan)


<a id="sec6"></a>
## 6. groupby & aggregation

Split rows by a key, compute a stat per group.

In [ ]:
# one stat per group
df.groupby('boatClass')['price'].median().sort_values(ascending=False)

# multiple stats at once
df.groupby('boatClass')['price'].agg(['median', 'mean', 'count'])

# different stats per column
df.groupby('boatClass').agg(median_price=('price', 'median'),
                            avg_hp=('totalHP', 'mean'),
                            n=('price', 'size'))

# group by two keys
df.groupby(['state', 'boatClass'])['price'].median()

# transform: broadcast group stat back to every row (same length as df)
df['class_median_price'] = df.groupby('boatClass')['price'].transform('median')
df['price_vs_class'] = df['price'] / df['class_median_price']   # relative to peers

age = (df.groupby('a')
           .apply(lambda x: pd.Series({'b': func1(x), 'c': len(x)}))
           .reset_index())

In [ ]:
# --- Bin a CONTINUOUS column into ranges, store it, then group by the bins ---
# Step 1: add a bucket column to the DataFrame with pd.cut
df['feature_bin'] = pd.cut(df['feature'], bins=np.arange(start, stop, step))

# Step 2: group by that bucket column like any other categorical
df.groupby('feature_bin')['value'].agg(['mean', 'size'])             # stats per bucket
df.groupby('feature_bin').agg(avg_val=('value', 'mean'),
                              n=('value', 'size'))                    # multi-col agg
df.groupby('feature_bin', observed=True)['value'].median()           # observed=True drops empty bins

# Compute a metric + count + numeric x-position per bin (ready to plot):
grp = (df.groupby('feature_bin')
         .apply(lambda x: pd.Series({'metric': metric_fn(x), 'n': len(x),
                                     'mid': x['feature'].mean()}))
         .reset_index())

# Tips:
#  - pd.cut  -> equal-WIDTH bins (fixed value ranges, like above)
#  - pd.qcut -> equal-COUNT bins (quantiles; same number of rows per bucket)
#       df['feature_q'] = pd.qcut(df['feature'], q=5, labels=['q1','q2','q3','q4','q5'])
#  - use x['feature'].mean() inside the group to recover a numeric x-position per bin
#    (the bin label is an Interval, not plottable on a numeric axis)


In [ ]:
train_df['totalHP'] = train_df.groupby(['boatClass', 'engineCategory'])['totalHP'].transform(lambda s: s.fillna(s.median()))
train_df['totalHP'] = train_df.groupby(['engineCategory'])['totalHP'].transform(lambda s: s.fillna(s.median()))
    

city2state = (train_df.dropna(subset=['city', 'state'])
                .groupby('city')['state']
                .agg(lambda s: s.mode().iloc[0]))
mask = test_df['state'].isna() & test_df['city'].notna()
test_df.loc[mask, 'state'] = test_df.loc[mask, 'city'].map(city2state)

<a id="sec7"></a>
## 7. Sorting & ranking

In [ ]:
df.sort_values('price', ascending=False).head(10)        # most expensive
df.sort_values(['state', 'price'], ascending=[True, False])  # multi-key
df.nlargest(10, 'price')[['make', 'model', 'price']]      # top 10 (faster than sort+head)
df.nsmallest(10, 'price')[['make', 'model', 'price']]     # bottom 10
df['price'].rank(ascending=False)                          # rank each row


<a id="sec8"></a>
## 8. Reusable helper functions

Drop these in any project.

In [ ]:
def overview(df):
    """Quick one-shot summary: shape, dtypes, missing %, cardinality."""
    out = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'non_null': df.notna().sum(),
        'missing_pct': (df.isna().mean() * 100).round(2),
        'distinct': df.nunique(),
        'sample_value': [df[c].dropna().iloc[0] if df[c].notna().any() else None
                         for c in df.columns],
    })
    print(f'shape: {df.shape}')
    return out.sort_values('missing_pct', ascending=False)


def top_values(df, col, n=10):
    """value_counts with count AND percent side by side."""
    vc = df[col].value_counts(dropna=False)
    return pd.DataFrame({'count': vc.head(n),
                         'pct': (vc.head(n) / len(df) * 100).round(2)})


def numeric_summary(df, cols=None):
    """describe + skew for numeric columns, transposed for readability."""
    num = df.select_dtypes('number') if cols is None else df[cols]
    s = num.describe(percentiles=[.01, .25, .5, .75, .99]).T
    s['skew'] = num.skew()
    return s.round(2)


def find_zeros(df):
    """Count zeros per numeric column — zeros often hide 'missing' values."""
    num = df.select_dtypes('number')
    z = (num == 0).sum()
    return z[z > 0].sort_values(ascending=False)


def filter_df(df, **conditions):
    """Filter by equality on multiple columns: filter_df(df, state='FL', type='power')."""
    mask = pd.Series(True, index=df.index)
    for col, val in conditions.items():
        mask &= (df[col].isin(val) if isinstance(val, (list, tuple, set))
                 else df[col] == val)
    return df[mask]


# usage:
# overview(df)
# top_values(df, 'boatClass')
# numeric_summary(df)
# find_zeros(df)
# filter_df(df, state=['FL', 'CA'], type='power')


<a id="sec9"></a>
## 9. Plot a group-level metric vs a feature

Template for the **Q5/Q6 pattern**: visualize how a *group-level* rate (one
computed from **summed** columns, e.g. prepayment **CPR**) varies with a feature.

Two reusable steps:
- **`metric_by_feature`** — group by the x-axis feature, compute the metric per
  group, and record each group's row count. Pass `bins=` for wide/continuous
  features (bin them first); omit it for discrete features with few values.
- **`plot_metric_vs_feature`** — scatter with **marker size ∝ group count**
  (so sparse, noisy buckets look small) plus a **centered rolling-mean** trend
  line that smooths bucket-to-bucket noise.

To reuse in another project, just replace the `smm`/`cpr` building blocks with
your own group-level metric and pass it as `metric_fn`.


In [ ]:
import matplotlib.pyplot as plt


# ---------------------------------------------------------------------------
# Step 1: aggregate the metric over a feature (x-axis).
#   - feature   : column to put on the x-axis
#   - metric_fn : function(sub_frame) -> scalar   (e.g. cpr)
#   - bins      : pass an array to pd.cut WIDE/continuous features into ranges
#                 (use for things like 'moneyness'); omit for discrete features
#                 with few distinct values (like integer 'loanage').
# Returns a tidy frame: [feature, metric, loans], sorted by feature.
# ---------------------------------------------------------------------------
def metric_by_feature(df, feature, metric_fn, bins=None):
    g = df.copy()
    if bins is not None:
        g['_bin'] = pd.cut(g[feature], bins=bins)
        out = (g.groupby('_bin')
                 .apply(lambda s: pd.Series({'metric': metric_fn(s),
                                             'a': len(s),
                                             feature: s[feature].mean()}))
                 .reset_index(drop=True))
    else:
        out = (g.groupby(feature)
                 .apply(lambda s: pd.Series({'metric': metric_fn(s),
                                             'a': len(s)}))
                 .reset_index())
    return out.dropna(subset=[feature]).sort_values(feature)


# ---------------------------------------------------------------------------
# Step 2: plot it. Marker size encodes how many rows back each point (so you
# can see which points are well-supported); the red line is a centered
# rolling-mean trend that cancels bucket-to-bucket noise.
# ---------------------------------------------------------------------------
def plot_metric_vs_feature(agg, feature, ylabel, title=None, window=5):
    agg = agg.sort_values(feature)
    plt.figure(figsize=(9, 5))
    plt.scatter(agg[feature], agg['metric'], s=agg['loans'] * 2,
                alpha=0.6, edgecolor='k', linewidth=0.3,
                label='point size ∝ group count')
    plt.plot(agg[feature],
             agg['metric'].rolling(window, min_periods=1, center=True).mean(),
             color='crimson', lw=2, label=f'{window}-bucket rolling mean')
    plt.xlabel(feature); plt.ylabel(ylabel)
    plt.title(title or f'{ylabel} vs {feature}')
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


# ---- usage: DISCRETE feature (Q5 — CPR vs loan age) ----
# age = metric_by_feature(sep, 'loanage', cpr)
# plot_metric_vs_feature(age, 'loanage', title='CPR vs Loan Age')

# ---- usage: WIDE feature -> bin first (Q6 — CPR vs moneyness) ----
# mny = metric_by_feature(sep, 'moneyness', cpr, bins=np.arange(-1.5, 3.0, 0.25))
# plot_metric_vs_feature(mny, 'moneyness', title='CPR vs Moneyness')


<a id="sec10"></a>
## 10. Parsing dates

A column of date *strings* is just text until you convert it — `pd.to_datetime`
gives a real `datetime64` dtype, which unlocks the `.dt` accessor (year, month,
day, weekday) and correct sorting/resampling.

In [ ]:
# --- String -> datetime ---
df['date'] = pd.to_datetime(df['date'], format='%m/%d/%Y')   # explicit format = fast & unambiguous
df['date'] = pd.to_datetime(df['date'], format='mixed')      # irregular/mixed formats (pandas >=2.0)
df['date'] = pd.to_datetime(df['date'], errors='coerce')     # unparseable -> NaT instead of error
# format codes: %Y 4-digit yr | %y 2-digit yr | %m month | %d day | %H:%M:%S time

# Extract parts via the .dt accessor (datetime dtype only)
df['year']    = df['date'].dt.year
df['month']   = df['date'].dt.month
df['day']     = df['date'].dt.day
df['weekday'] = df['date'].dt.day_name()

# Sanity check: day-of-month should span 1..31. A max of 12 means month/day got swapped.
df['date'].dt.day.value_counts().sort_index()

# Count rows that failed to parse (only meaningful with errors='coerce')
print('unparsed dates (NaT):', df['date'].isna().sum())

# NOTE: the old infer_datetime_format=True is deprecated -> use format='mixed' instead.


<a id="sec11"></a>
## 11. Scaling & normalization

Two different things:
- **Scaling** — squash values into a fixed *range* (e.g. 0–1) or to mean 0 / std 1.
  The distribution *shape* is unchanged. Needed by distance/gradient models (KNN, SVM, NN).
- **Normalization** — reshape the *distribution* itself toward a bell curve
  (e.g. Box-Cox). Needed by methods that assume normally-distributed data.

> Leakage warning: **fit the scaler on TRAIN only**, then `transform` train and test
> with it. Fitting on the full dataset leaks test statistics into training.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from scipy import stats

# --- Scaling ---
# Min-max -> [0, 1]   (note: pass a 2-D frame df[['col']], not a 1-D series)
df['price_scaled'] = MinMaxScaler().fit_transform(df[['price']])

# Standardize -> mean 0, std 1 (z-score)
df['price_z'] = StandardScaler().fit_transform(df[['price']])

# Proper train/test usage (fit on train only, no leakage):
# scaler = MinMaxScaler().fit(train[['price']])
# train['price_scaled'] = scaler.transform(train[['price']])
# test['price_scaled']  = scaler.transform(test[['price']])

# --- Normalization (reshape toward Gaussian) ---
# Box-Cox needs strictly positive values
pos = df['price'] > 0
df.loc[pos, 'price_boxcox'] = stats.boxcox(df.loc[pos, 'price'])[0]

# Log transform — quick, robust normalizer for right-skewed money/size data
df['price_log'] = np.log1p(df['price'])   # log1p handles 0 safely

# (The Kaggle course used mlxtend.preprocessing.minmax_scaling + scipy.stats.boxcox;
#  sklearn's scalers above are the more widely-available equivalent.)


<a id="sec12"></a>
## 12. Character encodings

When `pd.read_csv` throws a `UnicodeDecodeError`, the file isn't UTF-8. Sniff the
real encoding from the raw bytes with `chardet`, read with it, then re-save as UTF-8.

In [ ]:
import chardet

# 1) Detect encoding from a SAMPLE of raw bytes (reading all of a big file is slow)
with open('file.csv', 'rb') as f:
    guess = chardet.detect(f.read(100_000))
print(guess)        # e.g. {'encoding': 'Windows-1252', 'confidence': 0.73, ...}

# 2) Read using the detected encoding
df = pd.read_csv('file.csv', encoding=guess['encoding'])
# Fallbacks if the guess is wrong / low confidence:
#   encoding='latin-1' (a.k.a. ISO-8859-1) never errors — good last resort
#   encoding='utf-8'   the default

# 3) Save back out as clean UTF-8 (pandas default), so the next read just works
df.to_csv('file-utf8.csv', index=False)

# Single-string encode/decode (bytes <-> str)
"résumé".encode('utf-8')                   # str  -> bytes:  b'r\xc3\xa9sum\xc3\xa9'
b'r\xc3\xa9sum\xc3\xa9'.decode('utf-8')     # bytes -> str:   'résumé'


<a id="sec13"></a>
## 13. Inconsistent text entry & fuzzy matching

Free-text categoricals (city, country, make) often have the same value typed many
ways: `' Germany'`, `'germany'`, `'Germny'`. First **normalize** (lowercase + strip),
then **fuzzy-match** near-duplicates to a single canonical spelling.

In [ ]:
# 1) Normalize: lowercase + strip whitespace (fixes ' Germany' vs 'germany')
df['make'] = df['make'].str.lower().str.strip()

# Inspect distinct values to spot remaining inconsistencies
sorted(df['make'].dropna().unique())

# 2) Fuzzy-match near-duplicates to a canonical spelling
from rapidfuzz import process, fuzz        # maintained, fast drop-in for the older fuzzywuzzy

# see what's "close" to a target value (score 0-100)
process.extract('sea ray', df['make'].unique(), scorer=fuzz.ratio, limit=10)

# replace every value close enough to `target` with `target`
def replace_matches(df, col, target, min_score=90):
    strings = df[col].dropna().unique()
    close = [s for s, score, _ in process.extract(
                target, strings, scorer=fuzz.ratio, limit=len(strings))
             if score >= min_score]
    df.loc[df[col].isin(close), col] = target
    return df

# replace_matches(df, 'make', 'sea ray')   # collapses 'searay', 'sea-ray', 'sea ray ' -> 'sea ray'

# (Kaggle taught the fuzzywuzzy package; rapidfuzz is its faster, actively-maintained
#  successor with the same process.extract / fuzz.ratio API.)
